# Double-drift: perception vs saccades (Lisi & Cavanagh 2015): analysis

Computes Experiment 1's measures from the runs' `*_trials.csv` files, with the functions in
`lisi_cavanagh2015.analysis` (each cites the paper's analysis it follows):

1. **Perceptual task**: the physical path orientation seen as vertical (the PSE of a
   cumulative Gaussian fitted by maximum likelihood), for each drift pairing.
2. **Saccade task**: latencies outside 100-600 ms excluded; the path orientation targeted by
   the saccades, recovered from a multivariate linear model of the saccade amplitudes; the
   effect of the internal motion (double-drift minus control).
3. **Orthogonal landing error**: the small shift of landings in the direction of the
   internal drift at saccade onset, and whether it grows along the path.
4. **Perceived tilt vs saccade endpoint deviation**: the two effects side by side.

Set `ANALYSIS_RUN` to a folder to search (a run folder, a subject folder or a data root);
by default the runs under this project's `data*/` folders are used. Data from
`--mode simulate` or `lisi_cavanagh2015.rehearse` come from a simulated observer: they test
the software and the analysis, not the illusion.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

from lisi_cavanagh2015 import analysis as A
from lisi_cavanagh2015.geometry import path_vectors

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
ROOT = Path(os.environ['ANALYSIS_RUN']) if os.environ.get('ANALYSIS_RUN') else None


def find_runs(task):
    roots = [ROOT] if ROOT else sorted(HERE.glob('data*'))
    files = [f for r in roots for f in Path(r).rglob('*_trials.csv') if f'task-{task}' in f.parent.name]
    return sorted(files, key=lambda p: p.stat().st_mtime)


def load(task):
    frames = []
    for f in find_runs(task):
        df = pd.read_csv(f)
        df['subject'] = next((p.name for p in f.parents if p.name.startswith('sub-')), 'unknown')
        df['run_folder'] = f.parent.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


perc = load('double-drift-perception')
sacc_all = load('double-drift-saccades')
print('perceptual trials:', len(perc), '| saccade trials:', len(sacc_all))

## 1. Perceptual task: the orientation seen as vertical

In [ ]:
pse = pd.DataFrame()
if len(perc):
    pse = pd.concat([A.perceptual_pse(g).assign(subject=s) for s, g in perc.groupby('subject')],
                    ignore_index=True)
    print(pse.round(2).to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2), sharey=True)
    done = perc[perc['outcome'].isin(['REPORTED_RIGHT', 'REPORTED_LEFT'])]
    for ax, side, color in zip(axes, ['left', 'right'], ['tab:blue', 'tab:red']):
        g = done[done['tilt_side'] == side]
        if g.empty:
            continue
        binned = g.groupby(g['path_angle_deg'].round(-1))['outcome']
        prop = binned.agg(lambda o: (o == 'REPORTED_RIGHT').mean())
        ax.scatter(prop.index, prop.values, s=10 + 3 * binned.size().values, color=color, alpha=0.6)
        fit = A.fit_cumulative_gaussian(g['path_angle_deg'], g['outcome'] == 'REPORTED_RIGHT')
        xs = np.linspace(-80, 80, 400)
        ax.plot(xs, norm.cdf((xs - fit.mu) / fit.sigma), color=color)
        ax.axvline(fit.mu, color=color, ls=':')
        ax.set(title=f'{side}-tilt stimulus: PSE {fit.mu:.1f} deg', xlabel='physical path angle (deg, + = right)')
    axes[0].set_ylabel('P("right")')
    plt.tight_layout()
else:
    print('no perceptual run found')

## 2. Saccade task: the path orientation targeted by the saccades

In [ ]:
orient = pd.DataFrame()
effects = pd.DataFrame()
if len(sacc_all):
    done = sacc_all[sacc_all['outcome'] == 'SACCADE']
    usable = A.usable_saccades(sacc_all)
    print(f'saccades: {len(done)}; excluded (latency <100 or >600 ms, or no settled landing): '
          f'{len(done) - len(usable)} ({100 * (1 - len(usable) / max(len(done), 1)):.1f}%); '
          f'mean latency of the rest {usable["saccade_latency_ms"].mean():.0f} ms')
    orient = pd.concat([A.recover_orientations(g).assign(subject=s)
                        for s, g in usable.groupby('subject')], ignore_index=True)
    effects = pd.concat([A.internal_motion_effect(g.drop(columns='subject')).assign(subject=s)
                         for s, g in orient.groupby('subject')], ignore_index=True)
    print(orient.round(2).to_string(index=False))
    print()
    print('Effect of the internal motion on the recovered orientation (double-drift minus control):')
    print(effects.round(2).to_string(index=False))
else:
    print('no saccade run found')

## 3. Orthogonal landing error: the local shift in the drift direction

In [ ]:
shift = {}
if len(sacc_all):
    errs = A.orthogonal_errors(A.usable_saccades(sacc_all))
    shift = A.orthogonal_shift(errs)
    for k, v in shift.items():
        print(f'{k}: {v:.3f}')
    if len(errs):
        fig, ax = plt.subplots(figsize=(4.5, 3.2))
        for d, color in [('leftward', 'tab:blue'), ('rightward', 'tab:red')]:
            g = errs[errs['drift_at_onset'] == d]
            ax.scatter(g['gabor_along_at_onset_dva'], g['orthogonal_error_dva'], s=8, color=color,
                       alpha=0.5, label=f'{d} drift at onset')
        ax.axhline(0, color='0.6', lw=0.8)
        ax.set(xlabel='Gabor position along the path at onset (dva)',
               ylabel='landing error across the path (dva)')
        ax.legend(fontsize=8)
        plt.tight_layout()

## 4. Perceived tilt vs saccade endpoint deviation

Left: how far the internal motion rotates the path for perception (the physical orientation
seen as vertical) and for the saccades (double-drift minus control recovered orientation).
Right: one participant's double-drift landings, with the physical paths and the perceived
(vertical) path.

In [ ]:
if len(pse) and len(effects):
    m = pse.merge(effects[['subject', 'tilt_side', 'effect_deg']], on=['subject', 'tilt_side'])
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    sides = ['left', 'right']
    x = np.arange(len(sides))
    perc_dev = [m.loc[m.tilt_side == s, 'pse_deg'].mean() for s in sides]
    sacc_dev = [m.loc[m.tilt_side == s, 'effect_deg'].mean() for s in sides]
    ax.bar(x - 0.18, perc_dev, 0.34, label='perception: path seen as vertical (deg)', color='0.35')
    ax.bar(x + 0.18, sacc_dev, 0.34, label='saccades: double-drift minus control (deg)', color='tab:orange')
    ax.axhline(0, color='k', lw=0.8)
    ax.set_xticks(x, [f'{s} tilt' for s in sides])
    ax.set_ylabel('deviation from the physical path (deg)')
    ax.legend(fontsize=8)
    subj = m['subject'].iloc[0]
    u = A.usable_saccades(sacc_all[sacc_all['subject'] == subj])
    for side, color in [('left', 'tab:blue'), ('right', 'tab:red')]:
        g = u[(u['tilt_side'] == side) & u['internal_motion'].astype(bool)]
        if g.empty:
            continue
        ax2.scatter(g['endpoint_x_dva'], g['endpoint_y_dva'], s=8, color=color, alpha=0.5)
        ang = float(g['path_angle_deg'].iloc[0])
        uvec, _ = path_vectors(ang)
        c = np.array([g['path_center_x_dva'].iloc[0], g['path_center_y_dva'].iloc[0]])
        half = float(g['path_length_dva'].iloc[0]) / 2
        ends = np.array([c - half * uvec, c + half * uvec])
        ax2.plot(ends[:, 0], ends[:, 1], color=color, lw=2, label=f'{side}: physical path {ang:.0f} deg')
    ax2.plot([10, 10], [-1.5, 1.5], color='k', ls='--', lw=1, label='perceived path (vertical)')
    ax2.set_aspect('equal')
    ax2.set(xlabel='x (dva)', ylabel='y (dva)', title=f'{subj}: double-drift landings')
    ax2.legend(fontsize=7)
    plt.tight_layout()
    print(m.round(2).to_string(index=False))
else:
    print('needs both a perceptual and a saccade run')

**Reading it.** The paper found the paths seen as vertical at about +49 and -52 deg, and no
effect of the internal motion on the saccades (t(5) = 0.97 and 0.70; Figure 1F), with a
0.67 dva rightward-minus-leftward shift of the landings across the path that did not grow
along it. Numbers from a simulated observer show only that this pipeline recovers what the
observer was programmed to do.